In [1]:
%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


# Для всех датасетов в начале файла разделить их на 
- A с фактором соседства с В
- B с фактором соседства с А
- A * B
### И обрабатывать по отдельнтости каждый своей моделью

Например, у Fligts и Operational есть фактор flight
нужно сделать 3 датафрейма
1. Flights с доп бинарным признаком "Есть ли данные обо мне в Operational"
2. Operational с доп бинарным признаком "Есть ли данные обо мне в Flights"
3. Их пересечение

Потом будут рассматриваться как отдельные датасеты

Также и для Boarding и для всех

По итогу выйдут 7 датасетов исходных с соседствами
и всевозможные их пересечения (попарные, по тройкам и так далее)

выделять пересечения в отдельный датасет можно лишь если > 1000

Pipeline

- нормализовать файлы [ ]
- обработать их группы [ ]

Группировать по тройкам и так далее УЖЕ В САМОМ КОНЦЕ (ПОКА НЕ ВЫПОЛНЕНО)
В БОАРДИНГ СЕРИЯ ПАСПОРТА (ОБЕ КОЛОНКИ) НАДО НОРМАЛИЗОВАТЬ !!!!!!!!!!!!!!!!!!!!!!

# ПЛАН

## 1. Выделить все датафреймы, где есть люди
## 2. Добавить людей в датафреймы

In [2]:
import pandas as pd

path = "../../resources/processed/temp/"

BoardingDF = pd.read_csv(f"{path}BoardingData_parsed.csv", sep="|", engine="pyarrow") # ЕСТЬ ЛЮДИ
FlightsDF = pd.read_csv(f"{path}json_parsed.csv", sep="|", engine="pyarrow")
OperationalDF = pd.read_csv(f"{path}yaml_parsed.csv", sep="|", engine="pyarrow")
SirenaDF = pd.read_csv(f"{path}sirena_parsed.csv", sep="|", engine="pyarrow") # ЕСТЬ ЛЮДИ
BoardingPassDF = pd.read_csv(f"{path}cleaned_boarding_pass.csv", sep="|", engine="pyarrow") # ЕСТЬ ЛЮДИ
LoyalityDF = pd.read_csv(f"{path}xml_parsed.csv", sep="|", engine="pyarrow") # ЕСТЬ ЛЮДИ
TimetableDF = pd.read_csv(f"{path}Skyteam_Timetable.csv", sep="|", engine="pyarrow")

In [3]:
display(TimetableDF)

,VALIDITY,DAYS,DEP_TIME,ARR_TIME,FLIGHT,AIRCRAFT,TRAVEL_TIME,FROM_CITY,FROM_CODE,TO_CITY,TO_CODE,OPERATED_BY
0,01 Nov - 31 Jan,1234567,06:00:00,07:25,KL1328,73W,1H25M,"Aalborg, Denmark",AAL,"Amsterdam , Netherlands",AMS,NaN
1,01 Nov - 31 Jan,1234567,12:10:00,13:35,KL1334,73W,1H25M,"Aalborg, Denmark",AAL,"Amsterdam , Netherlands",AMS,NaN
2,01 Nov - 23 Dec,1234567,18:15:00,19:35,KL1336,EQV,1H20M,"Aalborg, Denmark",AAL,"Amsterdam , Netherlands",AMS,NaN
3,26 Dec - 06 Jan,1 34567,18:15:00,19:35,KL1336,EQV,1H20M,"Aalborg, Denmark",AAL,"Amsterdam , Netherlands",AMS,NaN
4,07 Jan - 31 Jan,1234567,18:15:00,19:35,KL1336,EQV,1H20M,"Aalborg, Denmark",AAL,"Amsterdam , Netherlands",AMS,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1527636,01 Nov - 31 Jan,1234567,16:00:00,18:35,MF8639,737,2H35M,"Xiamen , China",XMN,"Zunyi , China",WMT,NaN
1527637,01 Nov - 31 Jan,1234567,16:00:00,18:35,MF8639,737,2H35M,"Xiamen , China",XMN,"Zunyi , China",WMT,NaN
1527638,04 Nov - 31 Jan,1234567,16:00:00,18:35,MF8639,737,2H35M,"Xiamen , China",XMN,"Zunyi , China",WMT,NaN
1527639,04 Nov - 31 Jan,1234567,16:00:00,18:35,MF8639,737,2H35M,"Xiamen , China",XMN,"Zunyi , China",WMT,NaN


# Пересечения

In [4]:
import pandas as pd


def prepare_intersection_datasets(df1, keys1, df2, keys2, col_name):
  df1_res = df1.copy()
  df2_res = df2.copy()

  df2_keys_unique = df2[keys2].drop_duplicates()
  if keys1 != keys2:
    df2_keys_unique = df2_keys_unique.rename(columns=dict(zip(keys2, keys1)))

  df1_res = df1_res.merge(
      df2_keys_unique, on=keys1, how="left", indicator=True
  )
  df1_res[col_name] = (df1_res["_merge"] == "both").astype('int8')
  df1_res = df1_res.drop(columns=["_merge"])

  df1_keys_unique = df1[keys1].drop_duplicates()
  if keys1 != keys2:
    df1_keys_unique = df1_keys_unique.rename(columns=dict(zip(keys1, keys2)))

  df2_res = df2_res.merge(
      df1_keys_unique, on=keys2, how="left", indicator=True
  )
  df2_res[col_name] = (df2_res["_merge"] == "both").astype('int8')
  df2_res = df2_res.drop(columns=["_merge"])

  if keys1 == keys2:
    df12 = pd.merge(df1, df2, on=keys1, how="inner")
  else:
    df2_temp = df2.rename(columns=dict(zip(keys2, keys1)))
    df12 = pd.merge(df1, df2_temp, on=keys1, how="inner")

  return df1_res, df2_res, df12

In [5]:
BoardingDF['TicketNumber'] = BoardingDF['TicketNumber'].astype(str)
BoardingPassDF['Номер_эл_билета'] = BoardingPassDF['Номер_эл_билета'].astype(str)


flights, boarding, fligts_boarding = prepare_intersection_datasets(
    FlightsDF, ['Flight', 'Date'], BoardingDF, ['FlightNumber', 'FlightDate'], "boardFlightMatch"
)

boarding, operational, boarding_operational = prepare_intersection_datasets(
    boarding,  ['FlightNumber', 'FlightDate'], OperationalDF, ['Flight', 'Date'], "OperatFlightMatch"
)

flights, operational, flights_operational = prepare_intersection_datasets(
    flights, ['Flight', 'Date'], operational, ['Flight', 'Date'], "FlightOperationalMatch"
)

# flights_operational, boarding_operational, flights_boarding_operational = prepare_intersection_datasets(
#     flights_operational, ['Flight', 'Date'], boarding_operational, ['FlightNumber', 'FlightDate'], "OperationalFlightsBoardingMatch"
# )

boardingPass, boarding, pass_boarding = prepare_intersection_datasets(
    BoardingPassDF, ['Номер_эл_билета'], boarding, ['TicketNumber'], "PassBoardingMatch"
)

sirena, boarding, sirena_boarding = prepare_intersection_datasets(
    SirenaDF, ['FlightCode', 'DepartDate'], boarding, ['FlightNumber', 'FlightDate'], "SirenaBoardingMatch"
)

sirena, flights, sirena_flights = prepare_intersection_datasets(
    sirena, ['FlightCode', 'DepartDate'], flights, ['Flight', 'Date'], "SirenaFlightsMatch"
)

loyalty, operational, loyalty_operational = prepare_intersection_datasets(
    LoyalityDF, ['Code', 'Date'], operational, ['Flight', 'Date'], "LoyaltyOperationalMatch"
)

loyalty, flights, loyalty_flights = prepare_intersection_datasets(
    loyalty, ['Code', 'Date'], flights, ['Flight', 'Date'], "LoyaltyFlightsMatch"
)

# timetable, flights, timetable_flights = prepare_intersection_datasets(
#     TimetableDF, ['FLIGHT'], flights, ['Flight'], "TimetableFlightsMatch"
# )

In [6]:
BoardingDF = boarding
FlightsDF = flights
OperationalDF = operational
SirenaDF = sirena
BoardingPassDF = boardingPass
LoyalityDF = loyalty
# TimetableDF = timetable

# Нормализация BoardingDF

In [7]:
display(BoardingDF)

,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,PassengerBirthDate,PassengerDocument,BookingCode,TicketNumber,Baggage,FlightDate,FlightTime,FlightNumber,CodeShare,Destination,boardFlightMatch,OperatFlightMatch,PassBoardingMatch,SirenaBoardingMatch
0,SAVELII,VIKTOROVICH,RUSANOV,Male,03/10/1983,2879 096860,FRNINO,6625956945991971,Transit,2017-03-22,06:05:00,SU1369,Own,Moscow,0,1,0,0
1,LEV,MARKOVICH,ISAEV,Male,12/13/1975,1788 173211,Not presented,1643715499224676,Registered,2017-03-18,22:10:00,SU1180,Own,Volgograd,1,1,1,0
2,NIKOLAI,I.,NIKOLSKII,Male,12/26/1990,4396 926588,VWNYGF,6247422701565929,Transit,2017-03-18,22:10:00,SU1180,Own,Volgograd,1,1,1,0
3,ANATOLII,PETROVICH,SHILOV,Male,05/24/1997,2595 919752,WQFFUE,Not presented,Registered,2017-03-18,22:10:00,SU1180,Own,Volgograd,1,1,0,0
4,MIROSLAVA,VIACHESLAVOVNA,SEMENOVA,Female,01/31/1976,6775 516990,Not presented,Not presented,Registered,2017-03-12,11:45:00,SU6284,Own,Moscow,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155142,RAMIL,S.,DOROFEEV,Male,02/14/1994,3256 077151,SNRBXU,8653871661040452,Delayed,2017-05-18,06:30:00,SU1158,Own,Rostov,1,1,0,0
155143,RUSLAN,KONSTANTINOVICH,CHISTIAKOV,Male,02/14/1975,3497 458395,Not presented,0223584175158947,NaN,2017-05-18,06:30:00,SU1158,Own,Rostov,1,1,0,0
155144,SOFIA,VALEREVNA,KALMYKOVA,Female,12/17/1972,6618 352713,Not presented,Not presented,Registered,2017-10-02,07:30:00,SU5614,Own,Petropavlovsk,1,1,0,0
155145,VIACHESLAV,F.,KONONOV,Male,05/20/2000,4567 242088,Not presented,0975693658645383,Registered,2017-09-16,07:00:00,SU1640,Own,Omsk,1,1,0,0


In [8]:
BoardingDF["PassengerFirstName"] = BoardingDF['PassengerFirstName'].map(
    BoardingDF['PassengerFirstName'].value_counts(normalize=True)
)
BoardingDF["PassengerSecondName"] = BoardingDF['PassengerSecondName'].map(
    BoardingDF['PassengerSecondName'].value_counts(normalize=True)
)
BoardingDF["PassengerLastName"] = BoardingDF['PassengerLastName'].map(
    BoardingDF['PassengerLastName'].value_counts(normalize=True)
)

BoardingDF["PassengerSex"] = BoardingDF["PassengerSex"].map({'Female': 0, 'Male': 1})

BoardingDF['PassengerBirthDate'] = pd.to_datetime(BoardingDF['PassengerBirthDate'], format='%m/%d/%Y')
BoardingDF['FlightDate'] = pd.to_datetime(BoardingDF['FlightDate'], format='%Y-%m-%d')
BoardingDF['PassengerAge'] = ((BoardingDF['FlightDate'] - BoardingDF['PassengerBirthDate']).dt.days // 365.25).astype(int)

passport_series = BoardingDF['PassengerDocument'].astype(str).str.slice(0, 4)
BoardingDF['PassportSeriesLeft'] = passport_series.str.slice(0, 2)
BoardingDF['PassportSeriesRight'] = passport_series.str.slice(2, 4)

BoardingDF["BookingCode"] = (BoardingDF['BookingCode'] != 'Not presented').astype('int8')

BoardingDF['Baggage'] = BoardingDF['Baggage'].fillna('No')

BoardingDF['BaggageTransit'] = (BoardingDF['Baggage'] == 'Transit').astype('int8')
BoardingDF['BaggageDelayed'] = (BoardingDF['Baggage'] == 'Delayed').astype('int8')
BoardingDF['BaggageRegistered'] = (BoardingDF['Baggage'] == 'Registered').astype('int8')

BoardingDF['Flight_DayOfWeek'] = BoardingDF['FlightDate'].dt.dayofweek
BoardingDF['Flight_Month'] = BoardingDF['FlightDate'].dt.month

ft = BoardingDF['FlightTime'].astype(str)

BoardingDF['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

BoardingDF["FlightNumber"] = BoardingDF['FlightNumber'].map(
    BoardingDF['FlightNumber'].value_counts(normalize=True)
)

BoardingDF["TicketNumber"] = (BoardingDF['TicketNumber'] != 'Not presented').astype(int)

BoardingDF["Destination"] = BoardingDF['Destination'].map(
    BoardingDF['Destination'].value_counts(normalize=True)
)

BoardingDF["CodeShare"] = (BoardingDF['CodeShare'] != 'Operated').astype(int)

BoardingDF["Document"] = BoardingDF["PassengerDocument"]

BoardingDF = BoardingDF.drop(columns=['PassengerBirthDate', 'FlightDate', 'PassengerDocument', 'Baggage', 'FlightDate'])

In [9]:
display(BoardingDF)
BoardingDF.to_csv("../resources/Boarding.csv", sep="|", index=False)

,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,BookingCode,TicketNumber,FlightTime,FlightNumber,CodeShare,Destination,...,SirenaBoardingMatch,PassengerAge,PassportSeriesLeft,PassportSeriesRight,BaggageTransit,BaggageDelayed,BaggageRegistered,Flight_DayOfWeek,Flight_Month,Document
0,0.004654,0.002662,0.000786,1,1,1,365,0.004428,1,0.567346,...,0,34,28,79,1,0,0,2,3,2879 096860
1,0.004673,0.002424,0.000857,1,0,1,1330,0.001476,1,0.008914,...,0,41,17,88,0,0,1,5,3,1788 173211
2,0.004834,0.044899,0.001225,1,1,1,1330,0.001476,1,0.008914,...,0,26,43,96,1,0,0,5,3,4396 926588
3,0.005343,0.002494,0.000851,1,1,0,1330,0.001476,1,0.008914,...,0,19,25,95,0,0,1,5,3,2595 919752
4,0.004944,0.002598,0.000838,0,0,0,705,0.002849,1,0.567346,...,0,41,67,75,0,0,1,6,3,6775 516990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155142,0.005027,0.045389,0.001141,1,1,1,390,0.001424,1,0.011273,...,0,23,32,56,0,1,0,3,5,3256 077151
155143,0.004918,0.002462,0.000709,1,0,1,390,0.001424,1,0.011273,...,0,42,34,97,0,0,0,3,5,3497 458395
155144,0.005292,0.002475,0.000928,0,0,0,450,0.001959,1,0.010848,...,0,44,66,18,0,0,1,0,10,6618 352713
155145,0.004589,0.009926,0.001044,1,0,1,420,0.001334,1,0.004460,...,0,17,45,67,0,0,1,5,9,4567 242088


# Нормализация FlightsDF

In [10]:
display(FlightsDF)

,Date,Codeshare,Flight,DepCity,DepAirport,DepCountry,ArrCity,ArrAirport,ArrCountry,boardFlightMatch,FlightOperationalMatch,SirenaFlightsMatch,LoyaltyFlightsMatch
0,2017-03-04,False,DL3377,Boston MA,BOS,United States of,Pittsburgh PA,PIT,United States of,0,1,0,0
1,2017-07-21,False,DL3630,Green Bay WI,GRB,United States of,Detroit MI,DTW,United States of,0,1,0,0
2,2017-03-22,False,DL5361,Pittsburgh PA,PIT,United States of,New York NY,LGA,United States of,0,1,0,0
3,2017-01-04,True,SU3702,Moscow,SVO,Russian Federation,Warsaw,WAW,Poland,0,1,0,0
4,2017-08-08,False,DL3412,Fayetteville AR,XNA,United States of,Atlanta GA,ATL,United States of,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308982,2017-12-15,False,DL3345,Washington,IAD,United States of,New York NY,JFK,United States of,0,0,0,0
1308983,2017-08-12,False,SU2014,Moscow,SVO,Russian Federation,Prague,PRG,Czech Republic,0,0,0,0
1308984,2017-12-29,False,DL2463,Atlanta GA,ATL,United States of,New York NY,LGA,United States of,0,0,0,0
1308985,2017-06-02,False,SU1463,Novosibirsk,OVB,Russian Federation,Moscow,SVO,Russian Federation,1,1,0,0


In [11]:
FlightsDF['Date'] = pd.to_datetime(FlightsDF['Date'])
FlightsDF['Flight_DayOfWeek'] = FlightsDF['Date'].dt.dayofweek
FlightsDF['Flight_Month'] = FlightsDF['Date'].dt.month

FlightsDF['Codeshare'] = FlightsDF['Codeshare'].astype(int)

FlightsDF["Flight"] = FlightsDF['Flight'].map(
    FlightsDF['Flight'].value_counts(normalize=True)
)

FlightsDF['Airport'] = FlightsDF['DepAirport'] + "_" + FlightsDF['ArrAirport']
FlightsDF["Airport"] = FlightsDF['Airport'].map(
    FlightsDF['Airport'].value_counts(normalize=True)
)

FlightsDF['City'] = FlightsDF['DepCity'] + "_" + FlightsDF['ArrCity']
FlightsDF["City"] = FlightsDF['City'].map(
    FlightsDF['City'].value_counts(normalize=True)
)

FlightsDF['Country'] = FlightsDF['DepCountry'] + "_" + FlightsDF['ArrCountry']
FlightsDF["Country"] = FlightsDF['Country'].map(
    FlightsDF['Country'].value_counts(normalize=True)
)

FlightsDF = FlightsDF.drop(columns=['Date', 'DepAirport', 'ArrAirport', 'DepCity', 'ArrCity', 'DepCountry', 'ArrCountry'])

In [12]:
display(FlightsDF)

,Codeshare,Flight,boardFlightMatch,FlightOperationalMatch,SirenaFlightsMatch,LoyaltyFlightsMatch,Flight_DayOfWeek,Flight_Month,Airport,City,Country
0,0,0.000083,0,1,0,0,5,3,0.000129,0.000129,0.129043
1,0,0.000058,0,1,0,0,4,7,0.000167,0.000167,0.129043
2,0,0.000032,0,1,0,0,2,3,0.000334,0.000474,0.129043
3,1,0.000186,0,1,0,0,2,1,0.000905,0.000905,0.000912
4,0,0.000062,0,1,0,1,1,8,0.000202,0.000202,0.129043
...,...,...,...,...,...,...,...,...,...,...,...
1308982,0,0.000080,0,0,0,0,4,12,0.000316,0.000723,0.129043
1308983,0,0.000168,0,0,0,0,5,8,0.003615,0.003615,0.005781
1308984,0,0.000024,0,0,0,0,4,12,0.000395,0.000827,0.129043
1308985,0,0.000342,1,1,0,0,4,6,0.001390,0.001390,0.119576


# Нормализация OperationalDF

In [13]:
display(OperationalDF)

,Date,Flight,From,To,Status,FF_Airline,FF_Number,Class,Fare,OperatFlightMatch,FlightOperationalMatch,LoyaltyOperationalMatch
0,2017-01-01,AF1145,SVO,CDG,LANDED,FB,520518073,Y,YRSTFN,0,1,0
1,2017-01-01,AF1844,SVO,CDG,LANDED,FB,18427809,Y,YGRPZR,0,1,1
2,2017-01-01,AF4855,SVO,CDG,LANDED,KE,993927664,Y,YRSTJQ,0,1,0
3,2017-01-01,AF4901,SVO,CDG,LANDED,SU,789402704,Y,YGRPAT,0,0,0
4,2017-01-01,AZ545,SVO,FCO,LANDED,KE,352734424,Y,YFLXKO,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1687048,2018-01-01,VN8067,SGN,VCS,LANDED,DT,182808803,Y,YGRPMT,0,1,0
1687049,2018-01-01,VN8067,SGN,VCS,LANDED,FB,88616357,Y,YFLXDJ,0,1,0
1687050,2018-01-01,VN813,SGN,REP,LANDED,DT,574004748,Y,YGRPAU,0,1,0
1687051,2018-01-01,VN8204,HAN,DIN,LANDED,KE,425304412,Y,YSTNDC,0,0,0


In [14]:
OperationalDF['Date'] = pd.to_datetime(OperationalDF['Date'])
OperationalDF['Flight_DayOfWeek'] = OperationalDF['Date'].dt.dayofweek
OperationalDF['Flight_Month'] = OperationalDF['Date'].dt.month

OperationalDF["Flight"] = OperationalDF['Flight'].map(
    OperationalDF['Flight'].value_counts(normalize=True)
)

OperationalDF['Airport'] = OperationalDF['From'] + "_" + OperationalDF['To']
OperationalDF["Airport"] = OperationalDF['Airport'].map(
    OperationalDF['Airport'].value_counts(normalize=True)
)

OperationalDF['ClassY'] = (OperationalDF['Class'] == 'Y').astype('int8')
OperationalDF['ClassJ'] = (OperationalDF['Class'] == 'J').astype('int8')
OperationalDF['ClassP'] = (OperationalDF['Class'] == 'P').astype('int8')

OperationalDF['FF_Airline_SU'] = (OperationalDF['FF_Airline'] == 'SU').astype('int8')
OperationalDF['FF_Airline_KE'] = (OperationalDF['FF_Airline'] == 'KE').astype('int8')
OperationalDF['FF_Airline_DT'] = (OperationalDF['FF_Airline'] == 'DT').astype('int8')

OperationalDF["FF_Number"] = OperationalDF['FF_Number'].map(
    OperationalDF['FF_Number'].value_counts(normalize=True)
)

OperationalDF["Fare"] = OperationalDF['Fare'].map(
    OperationalDF['Fare'].value_counts(normalize=True)
)

OperationalDF = OperationalDF.drop(columns=['Date', 'Status', 'From', 'To', 'Class', "FF_Airline"])

In [15]:
display(OperationalDF)

,Flight,FF_Number,Fare,OperatFlightMatch,FlightOperationalMatch,LoyaltyOperationalMatch,Flight_DayOfWeek,Flight_Month,Airport,ClassY,ClassJ,ClassP,FF_Airline_SU,FF_Airline_KE,FF_Airline_DT
0,0.000167,0.000015,0.000309,0,1,0,6,1,0.005871,1,0,0,0,0,0
1,0.000226,0.000005,0.000318,0,1,1,6,1,0.005871,1,0,0,0,0,0
2,0.000182,0.000011,0.000302,0,1,0,6,1,0.005871,1,0,0,0,1,0
3,0.000191,0.000008,0.000307,0,0,0,6,1,0.005871,1,0,0,1,0,0
4,0.000181,0.000002,0.000311,0,0,1,6,1,0.003165,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1687048,0.000027,0.000014,0.000346,0,1,0,0,1,0.000207,1,0,0,0,0,1
1687049,0.000027,0.000007,0.000301,0,1,0,0,1,0.000207,1,0,0,0,0,0
1687050,0.000024,0.000016,0.000295,0,1,0,0,1,0.000115,1,0,0,0,0,1
1687051,0.000037,0.000011,0.000315,0,0,0,0,1,0.000072,1,0,0,0,1,0


# Нормализация BoardingPassDF

In [16]:
display(BoardingPassDF)

,Очередь_регистрации,Класс_обслуживания,Номер_рейса,Город_вылета,Город_прилета,Код_аэропорта_вылета,Код_аэропорта_прилета,Дата_вылета,Время_вылета,PNR_бронь,Номер_эл_билета,Бонусная_карта,Фамилия,Имя,Отчество,PassBoardingMatch
0,32,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,DYELAO,7100246988860481,NaN,ZHDANOVA,LIDIIA,NaN,0
1,50,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,JWARRD,7608904175905333,NaN,SOKOLOV,SVIATOSLAV,NaN,0
2,32,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,SETJJH,527257587351371,NaN,SUKHANOV,SVIATOSLAV,NaN,0
3,77,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,MZZNJI,3696048365585673,NaN,ZELENIN,DENIS,NaN,0
4,87,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,NJNNPC,7470610229847684,NaN,OSTROVSKAIA,IRINA,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615766,54,Y,AZ7240,MILAN,JEDDAH,MXP,JED,2018-01-01,11:00:00,BAJUBQ,8673981301035301,NaN,KORNILOV,VLADIMIR,T,0
615767,9,Y,DL1161,DAYTON OH,ATLANTA GA,DAY,ATL,2018-01-01,14:17:00,SXQFLS,6057453743742357,NaN,SHESTAKOVA,NELLI,A,0
615768,47,P,SU1502,MOSCOW,TYUMEN,SVO,TJM,2018-01-01,22:30:00,VMNBHI,4424048314480896,NaN,KOLPAKOVA,MARIIA,NaN,1
615769,49,Y,GA312,JAKARTA,SURABAYA,CGK,SUB,2018-01-01,11:05:00,LTVLUP,3825472788916248,NaN,IGNATOVA,ALINA,NaN,0


In [17]:
BoardingPassDF['ClassY'] = (BoardingPassDF['Класс_обслуживания'] == 'Y').astype('int8')
BoardingPassDF['ClassJ'] = (BoardingPassDF['Класс_обслуживания'] == 'J').astype('int8')
BoardingPassDF['ClassP'] = (BoardingPassDF['Класс_обслуживания'] == 'P').astype('int8')

BoardingPassDF['Queue'] = pd.to_numeric(
    BoardingPassDF['Очередь_регистрации']
)

BoardingPassDF['Дата_вылета'] = pd.to_datetime(BoardingPassDF['Дата_вылета'])
BoardingPassDF['Flight_DayOfWeek'] = BoardingPassDF['Дата_вылета'].dt.dayofweek
BoardingPassDF['Flight_Month'] = BoardingPassDF['Дата_вылета'].dt.month

BoardingPassDF['City'] = BoardingPassDF['Город_вылета'] + "_" + BoardingPassDF['Город_прилета']
BoardingPassDF["City"] = BoardingPassDF['City'].map(
    BoardingPassDF['City'].value_counts(normalize=True)
)

BoardingPassDF["PassengerFirstName"] = BoardingPassDF['Имя'].map(
    BoardingPassDF['Имя'].value_counts(normalize=True)
)
BoardingPassDF["PassengerSecondName"] = BoardingPassDF['Фамилия'].map(
    BoardingPassDF['Фамилия'].value_counts(normalize=True)
)
BoardingPassDF["PassengerLastName"] = BoardingPassDF['Отчество'].notna().astype('int8')

BoardingPassDF['Airport'] = BoardingPassDF['Код_аэропорта_вылета'] + "_" + BoardingPassDF['Код_аэропорта_прилета']
BoardingPassDF["Airport"] = BoardingPassDF['Airport'].map(
    BoardingPassDF['Airport'].value_counts(normalize=True)
)

BoardingPassDF["Flight"] = BoardingPassDF['Номер_рейса'].map(
    BoardingPassDF['Номер_рейса'].value_counts(normalize=True)
)

ft = BoardingPassDF['Время_вылета'].astype(str)

BoardingPassDF['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

BoardingPassDF['BonusCard'] = BoardingPassDF['Бонусная_карта'].notna().astype('int8')

BoardingPassDF = BoardingPassDF.drop(columns=['Отчество', 'Время_вылета', 'Номер_рейса', 'Имя', 'Фамилия', 'Дата_вылета', 'Бонусная_карта', 'PNR_бронь', 'Очередь_регистрации', 'Класс_обслуживания', 'Код_аэропорта_прилета', 'Код_аэропорта_вылета', 'Город_вылета', 'Город_прилета', 'Номер_эл_билета' ])

In [18]:
display(BoardingPassDF)

,PassBoardingMatch,ClassY,ClassJ,ClassP,Queue,Flight_DayOfWeek,Flight_Month,City,PassengerFirstName,PassengerSecondName,PassengerLastName,Airport,Flight,FlightTime,BonusCard
0,0,1,0,0,32,6,1,0.002462,0.005379,0.000867,0,0.002462,0.000239,685,0
1,0,1,0,0,50,6,1,0.002462,0.005096,0.000805,0,0.002462,0.000239,685,0
2,0,1,0,0,32,6,1,0.002462,0.005096,0.000843,0,0.002462,0.000239,685,0
3,0,1,0,0,77,6,1,0.002462,0.005016,0.000919,0,0.002462,0.000239,685,0
4,0,1,0,0,87,6,1,0.002462,0.005059,0.000965,0,0.002462,0.000239,685,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615766,0,1,0,0,54,0,1,0.000133,0.004604,0.001171,1,0.000133,0.000065,660,0
615767,0,1,0,0,9,0,1,0.000049,0.005221,0.000963,1,0.000049,0.000024,857,0
615768,1,0,0,1,47,0,1,0.000698,0.004835,0.001099,0,0.000698,0.000200,1350,0
615769,0,1,0,0,49,0,1,0.000352,0.004838,0.001111,0,0.000352,0.000015,665,0


# Нормализация SirenaDF

In [19]:
display(SirenaDF)

,PaxName,PaxBirthDate,DepartDate,DepartTime,ArrivalDate,ArrivalTime,FlightCode,ShFrom,Dest,Code,...,TravelDoc,Seat,Meal,TrvCls,Fare,Baggage,PaxAdditionalInfo,AgentInfo,SirenaBoardingMatch,SirenaFlightsMatch
0,ОЗЕРОВ ИЛЬДАР ДАНИИЛОВИЧ,1999-05-15,2017-05-30,00:05:00,2017-05-30,08:05:00,SU1306NO,SVO,OVB,ZBQSPY,...,053270 N/A,NaN,J,J,GRPGN0PC,S,FF#SU 38116280,Go2See,0,0
1,КОЛОСОВ САМИР ТАМЕРЛАНОВИЧ,None,2017-12-27,02:15:00,2017-12-27,04:40:00,SU1323NO,MMK,SVO,MHPBBX,...,645520 N/A,KSML,Y,Y,RSTUQ,9,FF#FB 284903754,NaN,0,0
2,ИГНАТОВА СНЕЖАНА КОНСТАНТИНОВНА,None,2017-09-19,06:40:00,2017-09-19,07:45:00,SU1481NO,KJA,SVO,REDLVB,...,961316 N/A,NaN,Y,Y,STNJL,F,NaN,KupiBilet,0,0
3,ЖАРОВ ПЛАТОН АЛЬБЕРТОВИЧ,1999-05-02,2017-03-18,22:10:00,2017-03-19,01:05:00,SU1180NO,SVO,VOG,NSJNGQ,...,865148 N/A,NaN,J,J,FLXLS0PC,#,FF#FB 884556993,Travelgenio,0,0
4,НИКОЛЬСКИЙ НИКОЛАЙ ИГОРЕВИЧ,1990-12-26,2017-03-18,22:10:00,2017-03-19,01:05:00,SU1180NO,SVO,VOG,VWNYGF,...,926588 N/A,NaN,Y,Y,FLXPG,NaN,FF#SU 183142068,OZON.travel,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155752,ЕМЕЛЬЯНОВ МИРОСЛАВ СВЯТОСЛАВОВИЧ,None,2017-02-14,03:10:00,2017-02-14,04:00:00,SU1181NO,VOG,SVO,JYYXED,...,319326 N/A,STML,Y,Y,STNER,F,NaN,OneTwoTrip,0,0
155753,ТАРАСОВ АРТЕМИЙ ГЕРМАНОВИЧ,None,2017-02-14,03:10:00,2017-02-14,04:00:00,SU1181NO,VOG,SVO,XDKDZF,...,078129 N/A,NaN,Y,Y,FLXKV1PC,T,FF#DT 261604825,OZON.travel,0,0
155754,ГОРОХОВ БОРИС СВЯТОГОРОВИЧ,1970-10-28,2017-02-14,03:10:00,2017-02-14,04:00:00,SU1181NO,VOG,SVO,SOWGLC,...,760618 N/A,NaN,Y,Y,FLXBF,F,FF#FB 902862806,Aeroflot,0,0
155755,ДОРОФЕЕВ РАМИЛЬ СВЯТОСЛАВОВИЧ,1994-02-14,2017-05-18,06:30:00,2017-05-18,08:35:00,SU1158NO,SVO,ROV,SNRBXU,...,077151 N/A,NaN,Y,Y,GRPMD,#,NaN,NaN,0,0


In [20]:
translit_dict = {
    'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D', 
    'Е': 'E', 'Ё': 'E', 'Ж': 'ZH', 'З': 'Z', 'И': 'I', 
    'Й': 'I', 'К': 'K', 'Л': 'L', 'М': 'M', 'Н': 'N', 
    'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T', 
    'У': 'U', 'Ф': 'F', 'Х': 'KH', 'Ц': 'TS', 'Ч': 'CH', 
    'Ш': 'SH', 'Щ': 'SHCH', 'Ъ': '', 'Ы': 'Y', 'Ь': '', 
    'Э': 'E', 'Ю': 'IU', 'Я': 'IA',
    
    'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd', 
    'е': 'e', 'ё': 'e', 'ж': 'zh', 'з': 'z', 'и': 'i', 
    'й': 'i', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n', 
    'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 
    'у': 'u', 'ф': 'f', 'х': 'kh', 'ц': 'ts', 'ч': 'ch', 
    'ш': 'sh', 'щ': 'shch', 'ъ': '', 'ы': 'y', 'ь': '', 
    'э': 'e', 'ю': 'iu', 'я': 'ia'
}

translit_names = SirenaDF['PaxName'].apply(
    lambda x: ''.join(translit_dict.get(c, c) for c in str(x))
)

parts = translit_names.str.split(expand=True).reindex(columns=[0, 1, 2], fill_value='')

SirenaDF['LastName'] = parts[0]
SirenaDF['FirstName'] = parts[1]
SirenaDF['SecondName'] = parts[2]

SirenaDF['PaxName'] = SirenaDF['SecondName'] + " " + SirenaDF['FirstName'] + " " + SirenaDF['LastName']

SirenaDF["FirstName"] = SirenaDF['FirstName'].map(
    SirenaDF['FirstName'].value_counts(normalize=True)
)
SirenaDF["SecondName"] = SirenaDF['SecondName'].map(
    SirenaDF['SecondName'].value_counts(normalize=True)
)
SirenaDF["LastName"] = SirenaDF['LastName'].map(
    SirenaDF['LastName'].value_counts(normalize=True)
)

SirenaDF['HasBirthDate'] = SirenaDF['PaxBirthDate'].notnull().astype(int)

SirenaDF['PaxBirthDate'] = pd.to_datetime(SirenaDF['PaxBirthDate'], errors='coerce')
SirenaDF['DepartDate'] = pd.to_datetime(SirenaDF['DepartDate'], errors='coerce')

SirenaDF['PassengerAge'] = (
    (SirenaDF['DepartDate'] - SirenaDF['PaxBirthDate']).dt.days // 365.25
).fillna(-1).astype(float)

SirenaDF['Route'] = SirenaDF['ShFrom'].astype(str) + '_' + SirenaDF['Dest'].astype(str)
SirenaDF["Route"] = SirenaDF['Route'].map(
    SirenaDF['Route'].value_counts(normalize=True)
)

SirenaDF['Flight_DayOfWeek'] = SirenaDF['DepartDate'].dt.dayofweek
SirenaDF['Flight_Month'] = SirenaDF['DepartDate'].dt.month

dt_str = SirenaDF['DepartTime'].astype(str)
SirenaDF['DepartTimeMinutes'] = (
    dt_str.str.slice(0, 2).astype(int) * 60 +
    dt_str.str.slice(3, 5).astype(int)
)

SirenaDF["FlightCode"] = SirenaDF['FlightCode'].map(
    SirenaDF['FlightCode'].value_counts(normalize=True)
)

SirenaDF = pd.get_dummies(SirenaDF, columns=['TrvCls'], prefix='TrvCls', dtype=int)


SirenaDF['ClassY'] = (SirenaDF['Meal'] == 'Y').astype('int8')
SirenaDF['ClassJ'] = (SirenaDF['Meal'] == 'J').astype('int8')
SirenaDF['ClassP'] = (SirenaDF['Meal'] == 'P').astype('int8')

SirenaDF = SirenaDF.drop(columns=[
    'PaxBirthDate', 'DepartDate', 'DepartTime', 
    'ArrivalDate', 'ArrivalTime', 'ShFrom', 'Dest', 'Code', 'Meal', 'e-Ticket', 'TravelDoc','PaxAdditionalInfo','Fare',
    'Seat', 'Baggage', 'AgentInfo'
])

In [21]:
display(SirenaDF)

,PaxName,FlightCode,SirenaBoardingMatch,SirenaFlightsMatch,LastName,FirstName,SecondName,HasBirthDate,PassengerAge,Route,Flight_DayOfWeek,Flight_Month,DepartTimeMinutes,TrvCls_A,TrvCls_J,TrvCls_P,TrvCls_Y,ClassY,ClassJ,ClassP
0,DANIILOVICH ILDAR OZEROV,0.001457,0,0,0.001027,0.004764,0.004956,1,18.0,0.006131,1,5,5,0,1,0,0,0,1,0
1,TAMERLANOVICH SAMIR KOLOSOV,0.002863,0,0,0.001066,0.004905,0.005354,0,-1.0,0.011364,2,12,135,0,0,0,1,1,0,0
2,KONSTANTINOVNA SNEZHANA IGNATOVA,0.003544,0,0,0.001117,0.004783,0.004745,0,-1.0,0.013264,1,9,400,0,0,0,1,1,0,0
3,ALBERTOVICH PLATON ZHAROV,0.001528,0,0,0.000770,0.005091,0.005245,1,17.0,0.008937,5,3,1330,0,1,0,0,0,1,0
4,IGOREVICH NIKOLAI NIKOLSKII,0.001528,0,0,0.001342,0.005033,0.004526,1,26.0,0.008937,5,3,1330,0,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155752,SVIATOSLAVOVICH MIROSLAV EMELIANOV,0.002960,0,0,0.001066,0.004950,0.005226,0,-1.0,0.017502,1,2,190,0,0,0,1,1,0,0
155753,GERMANOVICH ARTEMII TARASOV,0.002960,0,0,0.000918,0.004757,0.005194,0,-1.0,0.017502,1,2,190,0,0,0,1,1,0,0
155754,SVIATOGOROVICH BORIS GOROKHOV,0.002960,0,0,0.000982,0.004989,0.005496,1,46.0,0.017502,1,2,190,0,0,0,1,1,0,0
155755,SVIATOSLAVOVICH RAMIL DOROFEEV,0.001419,0,0,0.001066,0.004867,0.005226,1,23.0,0.011537,3,5,390,0,0,0,1,1,0,0


In [22]:
SirenaDF.to_csv("../resources/sirena.csv", sep="|", index="false")

# Нормализация fligts_boarding

In [23]:
display(fligts_boarding)

,Date,Codeshare,Flight,DepCity,DepAirport,DepCountry,ArrCity,ArrAirport,ArrCountry,PassengerFirstName,...,PassengerLastName,PassengerSex,PassengerBirthDate,PassengerDocument,BookingCode,TicketNumber,Baggage,FlightTime,CodeShare,Destination
0,2017-01-04,False,SU5615,Petropavlovsk,PKC,Russian Federation,Vladivostok,VVO,Russian Federation,NAZAR,...,ZYKOV,Male,05/22/1986,0182 855187,WGMDZO,1566303258389717,Registered,14:05:00,Own,Vladivostok
1,2017-01-04,False,SU5615,Petropavlovsk,PKC,Russian Federation,Vladivostok,VVO,Russian Federation,MIROSLAV,...,KOCHETKOV,Male,11/01/1988,0285 025502,Not presented,Not presented,Transit,14:05:00,Own,Vladivostok
2,2017-01-04,False,SU5615,Petropavlovsk,PKC,Russian Federation,Vladivostok,VVO,Russian Federation,ADEL,...,GONCHAROV,Male,02/01/1970,1368 908524,Not presented,3878530953422390,NaN,14:05:00,Own,Vladivostok
3,2017-01-04,False,SU5615,Petropavlovsk,PKC,Russian Federation,Vladivostok,VVO,Russian Federation,KAMILLA,...,MINAEVA,Female,05/28/1990,0279 223443,Not presented,Not presented,Transit,14:05:00,Own,Vladivostok
4,2017-01-04,False,SU5615,Petropavlovsk,PKC,Russian Federation,Vladivostok,VVO,Russian Federation,NAZAR,...,LOPATIN,Male,08/14/1995,8626 031268,QLVFSK,2351816126785487,Registered,14:05:00,Own,Vladivostok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1524937,2017-02-08,False,SU5613,Magadan,GDX,Russian Federation,Khabarovsk,KHV,Russian Federation,ARINA,...,ANTIPOVA,Female,12/04/1999,2639 803206,RYZASC,6121634497176881,Transit,15:50:00,Own,Khabarovsk
1524938,2017-02-08,False,SU5613,Magadan,GDX,Russian Federation,Khabarovsk,KHV,Russian Federation,TIKHON,...,GERASIMOV,Male,07/16/1997,0861 575695,Not presented,2174585924906444,Transit,15:50:00,Own,Khabarovsk
1524939,2017-04-10,False,SU1462,Moscow,SVO,Russian Federation,Novosibirsk,OVB,Russian Federation,AIDAR,...,ISAEV,Male,10/22/1984,3330 955743,LDAQZY,Not presented,NaN,06:05:00,Own,Novosibirsk
1524940,2017-04-10,False,SU1462,Moscow,SVO,Russian Federation,Novosibirsk,OVB,Russian Federation,ALEKSANDR,...,GLUKHOV,Male,03/19/1987,8252 830505,Not presented,7059809708981557,NaN,06:05:00,Own,Novosibirsk


In [24]:
fligts_boarding['Date'] = pd.to_datetime(fligts_boarding['Date'])
fligts_boarding['Flight_DayOfWeek'] = fligts_boarding['Date'].dt.dayofweek
fligts_boarding['Flight_Month'] = fligts_boarding['Date'].dt.month

fligts_boarding['Codeshare'] = fligts_boarding['Codeshare'].astype(int)

fligts_boarding["Flight"] = fligts_boarding['Flight'].map(
    fligts_boarding['Flight'].value_counts(normalize=True)
)

fligts_boarding["PassengerFirstName"] = fligts_boarding['PassengerFirstName'].map(
    fligts_boarding['PassengerFirstName'].value_counts(normalize=True)
)

fligts_boarding["PassengerLastName"] = fligts_boarding['PassengerLastName'].map(
    fligts_boarding['PassengerLastName'].value_counts(normalize=True)
)

fligts_boarding["PassengerSecondName"] = fligts_boarding['PassengerSecondName'].map(
    fligts_boarding['PassengerSecondName'].value_counts(normalize=True)
)


fligts_boarding['Airport'] = fligts_boarding['DepAirport'] + "_" + fligts_boarding['ArrAirport']
fligts_boarding["Airport"] = fligts_boarding['Airport'].map(
    fligts_boarding['Airport'].value_counts(normalize=True)
)

fligts_boarding['City'] = fligts_boarding['DepCity'] + "_" + fligts_boarding['ArrCity']
fligts_boarding["City"] = fligts_boarding['City'].map(
    fligts_boarding['City'].value_counts(normalize=True)
)

fligts_boarding['Country'] = fligts_boarding['DepCountry'] + "_" + fligts_boarding['ArrCountry']
fligts_boarding["Country"] = fligts_boarding['Country'].map(
    fligts_boarding['Country'].value_counts(normalize=True)
)

fligts_boarding["TicketNumber"] = (fligts_boarding['TicketNumber'] != 'Not presented').astype('int8')
fligts_boarding["BookingCode"] = (fligts_boarding['BookingCode'] != 'Not presented').astype('int8')
fligts_boarding["CodeShare"] = (fligts_boarding['CodeShare'] != 'Own').astype('int8')

fligts_boarding['Baggage'] = fligts_boarding['Baggage'].fillna('No')

fligts_boarding['BaggageTransit'] = (fligts_boarding['Baggage'] == 'Transit').astype('int8')
fligts_boarding['BaggageDelayed'] = (fligts_boarding['Baggage'] == 'Delayed').astype('int8')
fligts_boarding['BaggageRegistered'] = (fligts_boarding['Baggage'] == 'Registered').astype('int8')


fligts_boarding["PassengerSex"] = fligts_boarding["PassengerSex"].map({'Female': 0, 'Male': 1})


fligts_boarding = fligts_boarding.drop(columns=['Destination', 'Baggage', 'PassengerBirthDate', 'Date', 'DepAirport', 'ArrAirport', 'DepCity', 'ArrCity', 'DepCountry', 'ArrCountry'])

In [25]:
display(fligts_boarding)

,Codeshare,Flight,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,PassengerDocument,BookingCode,TicketNumber,FlightTime,CodeShare,Flight_DayOfWeek,Flight_Month,Airport,City,Country,BaggageTransit,BaggageDelayed,BaggageRegistered
0,0,0.002588,0.005680,0.025417,0.000868,1,0182 855187,1,1,14:05:00,0,2,1,0.006047,0.006047,1.0,0,0,1
1,0,0.002588,0.004426,0.003150,0.001100,1,0285 025502,0,0,14:05:00,0,2,1,0.006047,0.006047,1.0,1,0,0
2,0,0.002588,0.004652,0.009947,0.001045,1,1368 908524,0,1,14:05:00,0,2,1,0.006047,0.006047,1.0,0,0,0
3,0,0.002588,0.005683,0.002588,0.001112,0,0279 223443,0,0,14:05:00,0,2,1,0.006047,0.006047,1.0,1,0,0
4,0,0.002588,0.005680,0.002958,0.001070,1,8626 031268,1,1,14:05:00,0,2,1,0.006047,0.006047,1.0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1524937,0,0.012230,0.004899,0.003047,0.000874,0,2639 803206,1,1,15:50:00,0,2,2,0.012230,0.012230,1.0,1,0,0
1524938,0,0.012230,0.005760,0.071943,0.001052,1,0861 575695,0,1,15:50:00,0,2,2,0.012230,0.012230,1.0,1,0,0
1524939,0,0.000388,0.005188,0.071943,0.000704,1,3330 955743,1,0,06:05:00,0,0,4,0.001201,0.001201,1.0,0,0,0
1524940,0,0.000388,0.005346,0.002277,0.000784,1,8252 830505,0,1,06:05:00,0,0,4,0.001201,0.001201,1.0,0,0,0


In [26]:
#fligts_boarding.to_csv("../resources/flights_boarding.csv", sep="|", index="false")

# Нормализация boarding_operational

In [27]:
boarding_operational["PassengerFirstName"] = boarding_operational['PassengerFirstName'].map(
    boarding_operational['PassengerFirstName'].value_counts(normalize=True)
)

boarding_operational["PassengerLastName"] = boarding_operational['PassengerLastName'].map(
    boarding_operational['PassengerLastName'].value_counts(normalize=True)
)

boarding_operational["PassengerSecondName"] = boarding_operational['PassengerSecondName'].map(
    boarding_operational['PassengerSecondName'].value_counts(normalize=True)
)

boarding_operational['Baggage'] = boarding_operational['Baggage'].fillna('No')

boarding_operational['BaggageTransit'] = (boarding_operational['Baggage'] == 'Transit').astype('int8')
boarding_operational['BaggageDelayed'] = (boarding_operational['Baggage'] == 'Delayed').astype('int8')
boarding_operational['BaggageRegistered'] = (boarding_operational['Baggage'] == 'Registered').astype('int8')

boarding_operational["PassengerSex"] = boarding_operational["PassengerSex"].map({'Female': 0, 'Male': 1})
boarding_operational["BookingCode"] = (boarding_operational['BookingCode'] != 'Not presented').astype('int8')

boarding_operational['PassengerBirthDate'] = pd.to_datetime(boarding_operational['PassengerBirthDate'], format='%m/%d/%Y')
boarding_operational['FlightDate'] = pd.to_datetime(boarding_operational['FlightDate'], format='%Y-%m-%d')
boarding_operational['PassengerAge'] = ((boarding_operational['FlightDate'] - boarding_operational['PassengerBirthDate']).dt.days // 365.25).astype(int)

ft = boarding_operational['FlightTime'].astype(str)

boarding_operational['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

boarding_operational['ClassY'] = (boarding_operational['Class'] == 'Y').astype('int8')
boarding_operational['ClassJ'] = (boarding_operational['Class'] == 'J').astype('int8')
boarding_operational['ClassP'] = (boarding_operational['Class'] == 'P').astype('int8')

boarding_operational["FlightNumber"] = boarding_operational['FlightNumber'].map(
    boarding_operational['FlightNumber'].value_counts(normalize=True)
)

boarding_operational['FF_Airline_SU'] = (boarding_operational['FF_Airline'] == 'SU').astype('int8')
boarding_operational['FF_Airline_KE'] = (boarding_operational['FF_Airline'] == 'KE').astype('int8')
boarding_operational['FF_Airline_DT'] = (boarding_operational['FF_Airline'] == 'DT').astype('int8')

boarding_operational["FF_Number"] = boarding_operational['FF_Number'].map(
    boarding_operational['FF_Number'].value_counts(normalize=True)
)

boarding_operational["Fare"] = boarding_operational['Fare'].map(
    boarding_operational['Fare'].value_counts(normalize=True)
)

boarding_operational['Airport'] = boarding_operational['From'] + "_" + boarding_operational['To']
boarding_operational["Airport"] = boarding_operational['Airport'].map(
    boarding_operational['Airport'].value_counts(normalize=True)
)

boarding_operational["FF_Number"] = boarding_operational['FF_Number'].map(
    boarding_operational['FF_Number'].value_counts(normalize=True)
)

boarding_operational['FF_Airline_SU'] = (boarding_operational['FF_Airline'] == 'SU').astype('int8')
boarding_operational['FF_Airline_KE'] = (boarding_operational['FF_Airline'] == 'KE').astype('int8')
boarding_operational['FF_Airline_DT'] = (boarding_operational['FF_Airline'] == 'DT').astype('int8')

boarding_operational["CodeShare"] = (boarding_operational['CodeShare'] != 'Own').astype('int8')
boarding_operational = boarding_operational.drop(columns=['FF_Number', 'FF_Airline', "Status", "From", "To", 'Destination', 'Class', 'TicketNumber', 'Baggage', 'PassengerBirthDate', 'FlightDate'])

In [28]:
print(boarding_operational.dtypes)
display(boarding_operational)

PassengerFirstName     float64
PassengerSecondName    float64
PassengerLastName      float64
PassengerSex             int64
PassengerDocument          str
BookingCode               int8
FlightTime               int64
FlightNumber           float64
CodeShare                 int8
boardFlightMatch          int8
Fare                   float64
BaggageTransit            int8
BaggageDelayed            int8
BaggageRegistered         int8
PassengerAge             int64
ClassY                    int8
ClassJ                    int8
ClassP                    int8
FF_Airline_SU             int8
FF_Airline_KE             int8
FF_Airline_DT             int8
Airport                float64
dtype: object


,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,PassengerDocument,BookingCode,FlightTime,FlightNumber,CodeShare,boardFlightMatch,...,BaggageDelayed,BaggageRegistered,PassengerAge,ClassY,ClassJ,ClassP,FF_Airline_SU,FF_Airline_KE,FF_Airline_DT,Airport
0,0.003946,0.002715,0.000601,1,2879 096860,1,365,0.006288,0,0,...,0,0,34,1,0,0,0,0,1,0.018583
1,0.005202,0.002059,0.000692,1,1788 173211,0,1330,0.000277,0,1,...,0,1,41,1,0,0,0,0,1,0.001776
2,0.005202,0.002059,0.000692,1,1788 173211,0,1330,0.000277,0,1,...,0,1,41,1,0,0,0,0,1,0.001776
3,0.005202,0.002059,0.000692,1,1788 173211,0,1330,0.000277,0,1,...,0,1,41,0,1,0,0,0,0,0.001776
4,0.005079,0.044760,0.001207,1,4396 926588,1,1330,0.000277,0,1,...,0,0,26,1,0,0,0,0,1,0.001776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975060,0.004830,0.002582,0.000734,1,3497 458395,0,390,0.000290,0,1,...,0,0,42,1,0,0,1,0,0,0.002179
1975061,0.004739,0.002394,0.001039,0,6618 352713,0,450,0.000597,0,1,...,0,1,44,1,0,0,0,1,0,0.001266
1975062,0.004476,0.010001,0.000786,1,4567 242088,0,420,0.000259,0,1,...,0,1,17,1,0,0,0,0,1,0.000918
1975063,0.004439,0.004952,0.000983,0,2861 762272,0,420,0.000727,0,1,...,0,0,33,1,0,0,0,0,0,0.006736


In [29]:
#boarding_operational.to_csv("../resources/boarding_operational.csv", sep="|", index="false")

In [30]:
display(pass_boarding)
print(pass_boarding.dtypes)

,Очередь_регистрации,Класс_обслуживания,Номер_рейса,Город_вылета,Город_прилета,Код_аэропорта_вылета,Код_аэропорта_прилета,Дата_вылета,Время_вылета,PNR_бронь,...,PassengerDocument,BookingCode,Baggage,FlightDate,FlightTime,FlightNumber,CodeShare,Destination,boardFlightMatch,OperatFlightMatch
0,81,Y,SU5642,KHABAROVSK,NOVOSIBIRSK,KHV,OVB,2017-01-01,19:25:00,UDXDFD,...,1481 672135,Not presented,Registered,2017-01-01,19:25:00,SU5642,Own,Novosibirsk,1,1
1,51,Y,SU1445,IRKUTSK,MOSCOW,IKT,SVO,2017-01-01,19:40:00,FQSXBE,...,9383 317316,Not presented,Delayed,2017-01-01,19:40:00,SU1445,Own,Moscow,1,1
2,40,Y,SU1445,IRKUTSK,MOSCOW,IKT,SVO,2017-01-01,19:40:00,XXFKOI,...,0993 717366,XXFKOI,Delayed,2017-01-01,19:40:00,SU1445,Own,Moscow,1,1
3,96,P,SU4605,KHABAROVSK,OKHA,KHV,OHH,2017-01-01,14:20:00,EYZSJI,...,0016 277417,Not presented,Registered,2017-01-01,14:20:00,SU4605,Operated,Okha,1,1
4,7,Y,SU1325,MURMANSK,MOSCOW,MMK,SVO,2017-01-01,19:15:00,UGYMLA,...,5640 823746,Not presented,Transit,2017-01-01,19:15:00,SU1325,Own,Moscow,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22007,78,Y,SU1425,CHELYABINSK,MOSCOW,CEK,SVO,2018-01-01,05:55:00,VVARFU,...,8585 022921,Not presented,Transit,2018-01-01,05:55:00,SU1425,Own,Moscow,1,1
22008,12,Y,SU1169,ROSTOV,MOSCOW,ROV,SVO,2018-01-01,21:00:00,EYZHKV,...,9734 349053,EYZHKV,NaN,2018-01-01,21:00:00,SU1169,Own,Moscow,0,1
22009,27,Y,SU1336,MOSCOW,ARKHANGELSK,SVO,ARH,2018-01-01,17:20:00,FRLDSZ,...,3112 262803,FRLDSZ,Registered,2018-01-01,17:20:00,SU1336,Own,Arkhangelsk,1,1
22010,10,A,SU1280,MOSCOW,KAZAN,SVO,KZN,2018-01-01,10:55:00,JNWDUC,...,8823 858714,JNWDUC,Registered,2018-01-01,10:55:00,SU1280,Own,Kazan,1,1


Очередь_регистрации       int64
Класс_обслуживания          str
Номер_рейса                 str
Город_вылета                str
Город_прилета               str
Код_аэропорта_вылета        str
Код_аэропорта_прилета       str
Дата_вылета              object
Время_вылета             object
PNR_бронь                   str
Номер_эл_билета             str
Бонусная_карта              str
Фамилия                     str
Имя                         str
Отчество                    str
PassengerFirstName          str
PassengerSecondName         str
PassengerLastName           str
PassengerSex                str
PassengerBirthDate          str
PassengerDocument           str
BookingCode                 str
Baggage                     str
FlightDate               object
FlightTime               object
FlightNumber                str
CodeShare                   str
Destination                 str
boardFlightMatch           int8
OperatFlightMatch          int8
dtype: object


In [31]:
pass_boarding["PassengerFirstName"] = pass_boarding['PassengerFirstName'].map(
    pass_boarding['PassengerFirstName'].value_counts(normalize=True)
)
pass_boarding["PassengerSecondName"] = pass_boarding['PassengerSecondName'].map(
    pass_boarding['PassengerSecondName'].value_counts(normalize=True)
)
pass_boarding["PassengerLastName"] = pass_boarding['PassengerLastName'].map(
    pass_boarding['PassengerLastName'].value_counts(normalize=True)
)

pass_boarding["PassengerSex"] = pass_boarding["PassengerSex"].map({'Female': 0, 'Male': 1})

pass_boarding['PassengerBirthDate'] = pd.to_datetime(pass_boarding['PassengerBirthDate'], errors='coerce')
pass_boarding['FlightDate'] = pd.to_datetime(pass_boarding['FlightDate'], errors='coerce')
pass_boarding['PassengerAge'] = ((pass_boarding['FlightDate'] - pass_boarding['PassengerBirthDate']).dt.days // 365.25).fillna(-1).astype(int)

passport_series = pass_boarding['PassengerDocument'].astype(str).str.slice(0, 4)
pass_boarding['PassportSeriesLeft'] = passport_series.str.slice(0, 2)
pass_boarding['PassportSeriesRight'] = passport_series.str.slice(2, 4)

pass_boarding["BookingCode"] = (pass_boarding['BookingCode'] != 'Not presented').astype('int8')
pass_boarding["TicketNumber"] = (pass_boarding['Номер_эл_билета'] != 'Not presented').astype(int)
pass_boarding["CodeShare"] = (pass_boarding['CodeShare'] != 'Operated').astype(int)

pass_boarding['Baggage'] = pass_boarding['Baggage'].fillna('No')
pass_boarding['BaggageTransit'] = (pass_boarding['Baggage'] == 'Transit').astype('int8')
pass_boarding['BaggageDelayed'] = (pass_boarding['Baggage'] == 'Delayed').astype('int8')
pass_boarding['BaggageRegistered'] = (pass_boarding['Baggage'] == 'Registered').astype('int8')

pass_boarding['Flight_DayOfWeek'] = pass_boarding['FlightDate'].dt.dayofweek
pass_boarding['Flight_Month'] = pass_boarding['FlightDate'].dt.month

ft = pass_boarding['FlightTime'].astype(str)
pass_boarding['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

pass_boarding["FlightNumber"] = pass_boarding['FlightNumber'].map(
    pass_boarding['FlightNumber'].value_counts(normalize=True)
)
pass_boarding["Destination"] = pass_boarding['Destination'].map(
    pass_boarding['Destination'].value_counts(normalize=True)
)

pass_boarding['ClassY'] = (pass_boarding['Класс_обслуживания'] == 'Y').astype('int8')
pass_boarding['ClassJ'] = (pass_boarding['Класс_обслуживания'] == 'J').astype('int8')
pass_boarding['ClassP'] = (pass_boarding['Класс_обслуживания'] == 'P').astype('int8')

pass_boarding['City'] = pass_boarding['Город_вылета'].astype(str) + "_" + pass_boarding['Город_прилета'].astype(str)
pass_boarding["City"] = pass_boarding['City'].map(
    pass_boarding['City'].value_counts(normalize=True)
)

pass_boarding['Airport'] = pass_boarding['Код_аэропорта_вылета'].astype(str) + "_" + pass_boarding['Код_аэропорта_прилета'].astype(str)
pass_boarding["Airport"] = pass_boarding['Airport'].map(
    pass_boarding['Airport'].value_counts(normalize=True)
)

pass_boarding['BonusCard'] = pass_boarding['Бонусная_карта'].notna().astype('int8')
pass_boarding['PassportSeriesLeft'] = pass_boarding['PassportSeriesLeft'].notna().astype('int8')
pass_boarding['PassportSeriesRight'] = pass_boarding['PassportSeriesRight'].notna().astype('int8')

pass_boarding["Document"] = pass_boarding["PassengerDocument"]

pass_boarding = pass_boarding.drop(columns=[
    'PassengerBirthDate', 'FlightDate', 'PassengerDocument', 'Baggage', 
    'Класс_обслуживания', 'Город_вылета', 'Город_прилета', 
    'Код_аэропорта_вылета', 'Код_аэропорта_прилета', 'PNR_бронь', 
    'Номер_эл_билета', 'Бонусная_карта', 'Фамилия', 'Имя', 'Отчество', 
    'Время_вылета', 'Номер_рейса', 'Дата_вылета'
], errors='ignore')

In [32]:
display(pass_boarding)
print(pass_boarding.dtypes)

,Очередь_регистрации,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,BookingCode,FlightTime,FlightNumber,CodeShare,Destination,...,BaggageRegistered,Flight_DayOfWeek,Flight_Month,ClassY,ClassJ,ClassP,City,Airport,BonusCard,Document
0,81,0.004952,0.002817,0.001090,0,0,1165,0.003634,1,0.009677,...,1,6,1,1,0,0,0.003634,0.003634,0,1481 672135
1,51,0.005588,0.024487,0.001181,0,0,1180,0.002862,1,0.572415,...,0,6,1,1,0,0,0.014265,0.014265,0,9383 317316
2,40,0.005179,0.043976,0.000636,0,1,1180,0.002862,1,0.572415,...,0,6,1,1,0,0,0.014265,0.014265,0,0993 717366
3,96,0.005224,0.070870,0.000863,1,0,860,0.003634,0,0.006269,...,1,6,1,0,0,1,0.006269,0.006269,0,0016 277417
4,7,0.004861,0.025940,0.001090,0,0,1155,0.003180,1,0.572415,...,0,6,1,1,0,0,0.011675,0.011675,0,5640 823746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22007,78,0.004634,0.002090,0.000909,1,0,355,0.002726,1,0.572415,...,0,0,1,1,0,0,0.017718,0.017718,0,8585 022921
22008,12,0.006542,0.044567,0.001045,1,1,1260,0.002544,1,0.572415,...,0,0,1,1,0,0,0.016945,0.016945,1,9734 349053
22009,27,0.004997,0.041841,0.001317,0,1,1040,0.001635,1,0.005815,...,1,0,1,1,0,0,0.005815,0.005815,0,3112 262803
22010,10,0.004043,0.002680,0.000727,1,1,655,0.001908,1,0.014083,...,1,0,1,0,0,0,0.014083,0.014083,0,8823 858714


Очередь_регистрации      int64
PassengerFirstName     float64
PassengerSecondName    float64
PassengerLastName      float64
PassengerSex             int64
BookingCode               int8
FlightTime               int64
FlightNumber           float64
CodeShare                int64
Destination            float64
boardFlightMatch          int8
OperatFlightMatch         int8
PassengerAge             int64
PassportSeriesLeft        int8
PassportSeriesRight       int8
TicketNumber             int64
BaggageTransit            int8
BaggageDelayed            int8
BaggageRegistered         int8
Flight_DayOfWeek         int32
Flight_Month             int32
ClassY                    int8
ClassJ                    int8
ClassP                    int8
City                   float64
Airport                float64
BonusCard                 int8
Document                   str
dtype: object


In [33]:
#pass_boarding.to_csv("../resources/pass_boarding.csv", sep="|", index="false")

In [34]:
display(loyalty_operational)

,UserId,FirstName,LastName,CardNumber,BonusProgram,Code,Date,Departure,Arrival,Fare_x,From,To,Status,FF_Airline,FF_Number,Class,Fare_y,OperatFlightMatch,FlightOperationalMatch
0,613142142,IAROMIR,ZVEREV,FB 171388778,Flying Blue,KE827,2017-08-06,rea,SZX,YGRPZT,rea,SZX,LANDED,FB,171388778,Y,YGRPZT,0,0
1,613142142,IAROMIR,ZVEREV,FB 171388778,Flying Blue,MU9706,2017-10-26,PEK,BSD,YSTNYV,PEK,KMG,LANDED,FB,171388778,Y,YSTNYV,0,1
2,613142142,IAROMIR,ZVEREV,FB 171388778,Flying Blue,MU9706,2017-10-26,PEK,BSD,YSTNYV,PEK,KMG,LANDED,FB,936803748,Y,YSTNHR,0,1
3,613142142,IAROMIR,ZVEREV,FB 171388778,Flying Blue,MU9706,2017-10-26,PEK,BSD,YSTNYV,PEK,KMG,LANDED,SU,822779361,Y,YRSTTA,0,1
4,613142142,IAROMIR,ZVEREV,FB 171388778,Flying Blue,MU9706,2017-10-26,PEK,BSD,YSTNYV,PEK,KMG,LANDED,SU,842775631,Y,YRSTNN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1312850,293103570,EMIL,KAZAKOV,SU 856232932,Aeroflot Bonus,AR1142,2017-12-14,EZE,FCO,PRSTQA,EZE,FCO,LANDED,KE,637084941,P,PRSTQA,0,0
1312851,293103570,EMIL,KAZAKOV,SU 856232932,Aeroflot Bonus,AR1143,2017-09-16,FCO,EZE,YGRPFA,FCO,EZE,LANDED,DT,269347755,Y,YGRPFA,0,0
1312852,265781820,UL'YANA,KONONOVA,DT 302785701,Delta SkyMiles,GA502,2017-05-12,CGK,PNK,YFLXAU,CGK,PNK,LANDED,DT,302785701,Y,YFLXAU,0,0
1312853,265781820,UL'YANA,KONONOVA,DT 302785701,Delta SkyMiles,GA88,2017-08-02,CGK,AMS,YRSTQC,CGK,AMS,LANDED,DT,302785701,Y,YRSTQC,0,1


In [35]:
loyalty_operational['ClassY'] = (loyalty_operational['Class'] == 'Y').astype('int8')
loyalty_operational['ClassJ'] = (loyalty_operational['Class'] == 'J').astype('int8')
loyalty_operational['ClassP'] = (loyalty_operational['Class'] == 'P').astype('int8')

loyalty_operational["Name"] = loyalty_operational["FirstName"] + " " + loyalty_operational["LastName"]

loyalty_operational["FirstName"] = loyalty_operational['FirstName'].map(
    loyalty_operational['FirstName'].value_counts(normalize=True)
)
loyalty_operational["LastName"] = loyalty_operational['LastName'].map(
    loyalty_operational['LastName'].value_counts(normalize=True)
)

loyalty_operational['BonusProgramA'] = (loyalty_operational['BonusProgram'] == 'Aeroflot Bonus').astype('int8')
loyalty_operational['BonusProgramD'] = (loyalty_operational['BonusProgram'] == 'Delta SkyMiles').astype('int8')
loyalty_operational['BonusProgramK'] = (loyalty_operational['BonusProgram'] == 'Korean Air SKYPASS').astype('int8')

loyalty_operational['Date'] = pd.to_datetime(loyalty_operational['Date'])
loyalty_operational['Flight_DayOfWeek'] = loyalty_operational['Date'].dt.dayofweek
loyalty_operational['Flight_Month'] = loyalty_operational['Date'].dt.month

loyalty_operational['Airport'] = loyalty_operational['From'] + "_" + loyalty_operational['To']
loyalty_operational["Airport"] = loyalty_operational['Airport'].map(
    loyalty_operational['Airport'].value_counts(normalize=True)
)

loyalty_operational['ClassY'] = (loyalty_operational['Class'] == 'Y').astype('int8')
loyalty_operational['ClassJ'] = (loyalty_operational['Class'] == 'J').astype('int8')
loyalty_operational['ClassP'] = (loyalty_operational['Class'] == 'P').astype('int8')

loyalty_operational['FF_Airline_SU'] = (loyalty_operational['FF_Airline'] == 'SU').astype('int8')
loyalty_operational['FF_Airline_KE'] = (loyalty_operational['FF_Airline'] == 'KE').astype('int8')
loyalty_operational['FF_Airline_DT'] = (loyalty_operational['FF_Airline'] == 'DT').astype('int8')

loyalty_operational["FF_Number"] = loyalty_operational['FF_Number'].map(
    loyalty_operational['FF_Number'].value_counts(normalize=True)
)

loyalty_operational["Departure"] = loyalty_operational['Departure'].map(
    loyalty_operational['Departure'].value_counts(normalize=True)
)

loyalty_operational["Arrival"] = loyalty_operational['Arrival'].map(
    loyalty_operational['Arrival'].value_counts(normalize=True)
)

loyalty_operational["Fare_x"] = loyalty_operational['Fare_x'].map(
    loyalty_operational['Fare_x'].value_counts(normalize=True)
)

loyalty_operational = loyalty_operational.drop(columns=[
    'Class', 'UserId', 'BonusProgram', 'CardNumber', 'Code', 'Date', 'Status', 'FF_Airline', 'Fare_x', 'Fare_y', 'From', 'To' 
], errors='ignore')

In [36]:
display(loyalty_operational)

,FirstName,LastName,Departure,Arrival,FF_Number,OperatFlightMatch,FlightOperationalMatch,ClassY,ClassJ,ClassP,Name,BonusProgramA,BonusProgramD,BonusProgramK,Flight_DayOfWeek,Flight_Month,Airport,FF_Airline_SU,FF_Airline_KE,FF_Airline_DT
0,0.002198,0.001012,0.008059,0.001961,2.056587e-05,0,0,1,0,0,IAROMIR ZVEREV,0,0,0,6,8,0.000021,0,0,0
1,0.002198,0.001012,0.003454,0.000070,2.056587e-05,0,1,1,0,0,IAROMIR ZVEREV,0,0,0,3,10,0.000195,0,0,0
2,0.002198,0.001012,0.003454,0.000070,9.140385e-06,0,1,1,0,0,IAROMIR ZVEREV,0,0,0,3,10,0.000195,0,0,0
3,0.002198,0.001012,0.003454,0.000070,6.855289e-06,0,1,1,0,0,IAROMIR ZVEREV,0,0,0,3,10,0.000195,1,0,0
4,0.002198,0.001012,0.003454,0.000070,5.331891e-06,0,1,1,0,0,IAROMIR ZVEREV,0,0,0,3,10,0.000195,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1312850,0.002181,0.001111,0.000778,0.012053,2.285096e-06,0,0,0,0,1,EMIL KAZAKOV,1,0,0,3,12,0.000036,0,1,0
1312851,0.002181,0.001111,0.009756,0.000839,4.570192e-06,0,0,1,0,0,EMIL KAZAKOV,1,0,0,5,9,0.000104,0,0,1
1312852,0.002204,0.000833,0.002159,0.000053,2.285096e-06,0,0,1,0,0,UL'YANA KONONOVA,0,1,0,4,5,0.000030,0,0,1
1312853,0.002204,0.000833,0.002159,0.012888,2.285096e-06,0,1,1,0,0,UL'YANA KONONOVA,0,1,0,2,8,0.000036,0,0,1


In [37]:
#loyalty_operational.to_csv("../resources/loyalty_operational.csv", sep="|", index="false")

In [38]:
display(loyalty_flights)

,UserId,FirstName,LastName,CardNumber,BonusProgram,Code,Date,Departure,Arrival,Fare,...,Codeshare,DepCity,DepAirport,DepCountry,ArrCity,ArrAirport,ArrCountry,boardFlightMatch,FlightOperationalMatch,SirenaFlightsMatch
0,613142142,IAROMIR,ZVEREV,FB 171388778,Flying Blue,MU9706,2017-10-26,PEK,BSD,YSTNYV,...,False,Baoshan,BSD,China,Kunming,KMG,China,0,1,0
1,103197717,VITALINA,KOROVINA,KE 696768759,Korean Air SKYPASS,SU1523,2017-02-12,NUX,SVO,YFLXPS,...,False,Moscow,SVO,Russian Federation,Novyj Urengoj,NUX,Russian Federation,1,1,0
2,103197717,VITALINA,KOROVINA,KE 696768759,Korean Air SKYPASS,SU1523,2017-02-12,NUX,SVO,YFLXPS,...,False,Novyj Urengoj,NUX,Russian Federation,Moscow,SVO,Russian Federation,1,1,0
3,103197717,VITALINA,KOROVINA,KE 696768759,Korean Air SKYPASS,SU1523,2017-02-12,NUX,SVO,YFLXPS,...,False,Novyj Urengoj,NUX,Russian Federation,Moscow,SVO,Russian Federation,1,1,0
4,103197717,VITALINA,KOROVINA,KE 696768759,Korean Air SKYPASS,SU1523,2017-02-12,NUX,SVO,YFLXPS,...,False,Novyj Urengoj,NUX,Russian Federation,Moscow,SVO,Russian Federation,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899931,293103570,EMIL,KAZAKOV,KE 637084941,Korean Air SKYPASS,SU6401,2017-02-14,LED,SVX,YFLXZR,...,False,Saint Petersburg,LED,Russian,Yekaterinburg,SVX,Russian Federation,0,1,0
899932,293103570,EMIL,KAZAKOV,DT 269347755,Delta SkyMiles,SU6402,2017-03-01,SVX,LED,YGRPPM,...,False,Yekaterinburg,SVX,Russian Federation,Saint Petersburg,LED,Russian,0,1,0
899933,293103570,EMIL,KAZAKOV,DT 269347755,Delta SkyMiles,SU6402,2017-03-01,SVX,LED,YGRPPM,...,False,Yekaterinburg,SVX,Russian Federation,Saint Petersburg,LED,Russian,0,1,0
899934,293103570,EMIL,KAZAKOV,DT 269347755,Delta SkyMiles,SU6402,2017-03-01,SVX,LED,YGRPPM,...,False,Yekaterinburg,SVX,Russian Federation,Saint Petersburg,LED,Russian,0,1,0


In [39]:
import pandas as pd

loyalty_flights["Name"] = loyalty_flights["FirstName"] + " " + loyalty_flights["LastName"]

loyalty_flights["FirstName"] = loyalty_flights['FirstName'].map(
    loyalty_flights['FirstName'].value_counts(normalize=True)
)
loyalty_flights["LastName"] = loyalty_flights['LastName'].map(
    loyalty_flights['LastName'].value_counts(normalize=True)
)

loyalty_flights['BonusProgramA'] = (loyalty_flights['BonusProgram'] == 'Aeroflot Bonus').astype('int8')
loyalty_flights['BonusProgramD'] = (loyalty_flights['BonusProgram'] == 'Delta SkyMiles').astype('int8')
loyalty_flights['BonusProgramK'] = (loyalty_flights['BonusProgram'] == 'Korean Air SKYPASS').astype('int8')
loyalty_flights['BonusProgramFB'] = (loyalty_flights['BonusProgram'] == 'Flying Blue').astype('int8')

loyalty_flights['Date'] = pd.to_datetime(loyalty_flights['Date'], errors='coerce')
loyalty_flights['Flight_DayOfWeek'] = loyalty_flights['Date'].dt.dayofweek
loyalty_flights['Flight_Month'] = loyalty_flights['Date'].dt.month

loyalty_flights['Codeshare'] = loyalty_flights['Codeshare'].astype(int)

loyalty_flights['Airport'] = loyalty_flights['DepAirport'].astype(str) + "_" + loyalty_flights['ArrAirport'].astype(str)
loyalty_flights["Airport"] = loyalty_flights['Airport'].map(
    loyalty_flights['Airport'].value_counts(normalize=True)
)

loyalty_flights['City'] = loyalty_flights['DepCity'].astype(str) + "_" + loyalty_flights['ArrCity'].astype(str)
loyalty_flights["City"] = loyalty_flights['City'].map(
    loyalty_flights['City'].value_counts(normalize=True)
)

loyalty_flights['Country'] = loyalty_flights['DepCountry'].astype(str) + "_" + loyalty_flights['ArrCountry'].astype(str)
loyalty_flights["Country"] = loyalty_flights['Country'].map(
    loyalty_flights['Country'].value_counts(normalize=True)
)

for col in ['Departure', 'Arrival', 'Fare', 'CardNumber', 'Code']:
    if col in loyalty_flights.columns:
        loyalty_flights[col] = loyalty_flights[col].map(
            loyalty_flights[col].value_counts(normalize=True)
        )

loyalty_flights = loyalty_flights.drop(columns=[
    'Date', 'DepAirport', 'ArrAirport', 'DepCity', 'ArrCity', 
    'DepCountry', 'ArrCountry', 'UserId', 'BonusProgram', 'CardNumber', 'Code'
], errors='ignore')

In [40]:
display(loyalty_flights)

,FirstName,LastName,Departure,Arrival,Fare,LoyaltyOperationalMatch,Codeshare,boardFlightMatch,FlightOperationalMatch,SirenaFlightsMatch,Name,BonusProgramA,BonusProgramD,BonusProgramK,BonusProgramFB,Flight_DayOfWeek,Flight_Month,Airport,City,Country
0,0.002081,0.001072,0.002719,0.000058,0.000396,1,0,0,1,0,IAROMIR ZVEREV,0,0,0,1,3,10,0.000042,0.000042,0.040433
1,0.004793,0.000757,0.017341,0.410467,0.000530,1,0,1,1,0,VITALINA KOROVINA,0,0,1,0,6,2,0.001719,0.001719,0.558466
2,0.004793,0.000757,0.017341,0.410467,0.000530,1,0,1,1,0,VITALINA KOROVINA,0,0,1,0,6,2,0.017330,0.017330,0.558466
3,0.004793,0.000757,0.017341,0.410467,0.000530,1,0,1,1,0,VITALINA KOROVINA,0,0,1,0,6,2,0.017330,0.017330,0.558466
4,0.004793,0.000757,0.017341,0.410467,0.000530,1,0,1,1,0,VITALINA KOROVINA,0,0,1,0,6,2,0.017330,0.017330,0.558466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899931,0.002091,0.001057,0.018138,0.001939,0.000346,1,0,0,1,0,EMIL KAZAKOV,0,0,1,0,1,2,0.000228,0.000228,0.014948
899932,0.002091,0.001057,0.004569,0.034920,0.000346,1,0,0,1,0,EMIL KAZAKOV,0,1,0,0,2,3,0.000296,0.000296,0.035889
899933,0.002091,0.001057,0.004569,0.034920,0.000346,1,0,0,1,0,EMIL KAZAKOV,0,1,0,0,2,3,0.000296,0.000296,0.035889
899934,0.002091,0.001057,0.004569,0.034920,0.000346,1,0,0,1,0,EMIL KAZAKOV,0,1,0,0,2,3,0.000296,0.000296,0.035889


In [41]:
loyalty_flights.to_csv("../resources/loyalty_flights.csv", sep="|", index="false")